In [ ]:
# ================================
# 07-hybrid-experiment.ipynb
# Phase 5: Hybrid IA³ + LoRA sequential adaptation
# Fix: uninstall incompatible torchao first
# ================================

# ------------------------------
# Fix environment: remove incompatible torchao
# ------------------------------
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q trl --no-deps
!pip install -q accelerate

import torch, transformers, datasets, peft
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Quick CUDA sanity check
x = torch.randn(100).cuda()
y = torch.randn(100).cuda()
z = torch.matmul(x, y)
print("✓ CUDA ops work:", z.item())

# ------------------------------
# Imports
# ------------------------------
import os
import time
import json
import math
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, IA3Config, TaskType, get_peft_model
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------
# Configuration
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
SEEDS = [42, 123, 456]
BUDGETS = [100, 500, 2000]          # critical budgets
LANGUAGES = ["hi"]                  # start with Hindi; add "te" later
HYBRID_LRS = [1e-4, 5e-4, 1e-3]     # learning rates to test

# Paths (same as previous notebooks)
DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ------------------------------
# Helper functions (reused from prior notebooks)
# ------------------------------
def load_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(language, budget):
    train_df = pd.read_parquet(f"{DATA_ROOT}/{language}/train_{budget}.parquet")
    valid_df = pd.read_parquet(f"{DATA_ROOT}/{language}/valid.parquet")

    def tokenize(batch):
        return tokenizer(
            batch["premise"], batch["hypothesis"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH
        )

    train_ds = Dataset.from_pandas(train_df).rename_column("label", "labels")
    valid_ds = Dataset.from_pandas(valid_df).rename_column("label", "labels")
    train_ds = train_ds.map(tokenize, batched=True)
    valid_ds = valid_ds.map(tokenize, batched=True)

    keep = ["input_ids", "attention_mask", "labels"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
    valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in keep])
    train_ds.set_format("torch")
    valid_ds.set_format("torch")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=32)
    return train_loader, valid_loader

# ------------------------------
# Hybrid model builder: sequential IA³ → LoRA
# ------------------------------
def build_hybrid_model():
    base = load_base_model()

    # 1. Apply IA³
    ia3_config = IA3Config(
        task_type=TaskType.SEQ_CLS,
        target_modules=["key", "value", "output.dense"],
        feedforward_modules=["output.dense"],
        modules_to_save=["classifier"]          # keep classifier trainable
    )
    model = get_peft_model(base, ia3_config)

    # 2. Apply LoRA on top of IA³
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["query", "value"],     # overlaps with IA³ on "value" (intentional)
        modules_to_save=["classifier"]         # classifier already trainable; safe to repeat
    )
    model = get_peft_model(model, lora_config)
    return model

# ------------------------------
# Training and evaluation for one hybrid configuration
# ------------------------------
def train_and_evaluate_hybrid(method, language, budget, seed, lr, epochs):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    train_loader, valid_loader = build_loaders(language, budget)

    # Build hybrid model
    model = build_hybrid_model()
    trainable_params = count_trainable_params(model)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    # Compute steps for scheduler (warmup)
    steps_per_epoch = math.ceil(budget / BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(0.1 * total_steps)

    model.train()
    start_time = time.perf_counter()

    for epoch in range(epochs):
        for batch in train_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    train_time = time.perf_counter() - start_time

    # Peak GPU memory measurement (dummy forward)
    torch.cuda.reset_peak_memory_stats()
    model.train()
    with torch.no_grad():
        dummy_batch = next(iter(train_loader))
        dummy_batch = {k: v.cuda() for k, v in dummy_batch.items()}
        _ = model(**dummy_batch)
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

    # Evaluation
    model.eval()
    predictions, labels = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    # Clean up
    del model
    torch.cuda.empty_cache()

    return {
        "method": method,
        "language": language,
        "budget": budget,
        "seed": seed,
        "lr": lr,
        "epochs": epochs,
        "warmup_steps": warmup_steps,
        "total_steps": total_steps,
        "accuracy": round(accuracy, 6),
        "macro_f1": round(macro_f1, 6),
        "trainable_params": trainable_params,
        "peak_gpu_memory_gb": round(peak_memory_gb, 4),
        "training_time_sec": round(train_time, 2)
    }

# ------------------------------
# Main experiment loop
# ------------------------------
results_file = "/kaggle/working/hybrid_experiment_results.csv"
results = []

total_runs = len(LANGUAGES) * len(BUDGETS) * len(HYBRID_LRS) * len(SEEDS)
current_run = 0

for language in LANGUAGES:
    for budget in BUDGETS:
        # Epochs: 10 for budgets <= 500, else 5 (matches original sweep)
        epochs = 10 if budget <= 500 else 5
        for lr in HYBRID_LRS:
            for seed in SEEDS:
                current_run += 1
                print(f"\n[{current_run}/{total_runs}] Hybrid IA³+LoRA | {language} | budget={budget} | lr={lr} | seed={seed}")

                try:
                    result = train_and_evaluate_hybrid(
                        method="ia3_lora_sequential",
                        language=language,
                        budget=budget,
                        seed=seed,
                        lr=lr,
                        epochs=epochs
                    )
                    results.append(result)
                    print(f"  ✓ Acc: {result['accuracy']:.4f}  F1: {result['macro_f1']:.4f}  Time: {result['training_time_sec']:.1f}s")

                    # Save incrementally
                    pd.DataFrame([result]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

                except Exception as e:
                    print(f"  ✗ ERROR: {e}")
                    error_row = {
                        "method": "ia3_lora_sequential",
                        "language": language,
                        "budget": budget,
                        "seed": seed,
                        "lr": lr,
                        "error": str(e)
                    }
                    pd.DataFrame([error_row]).to_csv(results_file, mode='a', header=not os.path.exists(results_file), index=False)

print(f"\n✅ Hybrid experiment completed. {total_runs} configurations attempted.")

# ------------------------------
# Post‑run summary
# ------------------------------
if os.path.exists(results_file):
    df_results = pd.read_csv(results_file)
    print("\n=== Summary of Hybrid Results ===")
    summary = df_results.groupby(["language", "budget", "lr"]).agg({
        "macro_f1": ["mean", "std", "count"],
        "accuracy": ["mean", "std"]
    }).round(4)
    print(summary)

    # Identify collapsed runs (accuracy ~0.3333)
    collapsed = df_results[np.isclose(df_results["accuracy"], 1/3, atol=0.001)]
    print(f"\nCollapsed runs (accuracy = 0.3333): {len(collapsed)} / {len(df_results)}")
    if len(collapsed) > 0:
        print(collapsed[["language", "budget", "lr", "seed", "accuracy"]])
else:
    print("No results file found.")

print("\nAll done.")